# Experiments (DDP)
Ablations on top of phases 1–3, one per Kaggle session (pick `EXP` in the control panel). Functions live in `run_exp.py`.
Every arm is saved under `runs/<mode>/exp/<arm>/final` (adapter dir: `load_encoder` / `evaluate_model` accept it), with its
`log.jsonl`; all rows go to `runs/<mode>/exp/metrics.json` and plots to `runs/<mode>/exp/plots/`. Re-running resumes:
finished arms are not retrained and finished rows not re-evaluated. The teacher is frozen; its cached vectors are reused.

| `EXP` | Paper | What |
|---|---|---|
| `losses` | Table 5 | Phase 2 from scratch with L2 vs KL (KaLM-style). L2 stays the default everywhere else. |
| `oneshot` | Table 8 (proxy) | Fresh adapter trained only with contrastive loss (no alignment) vs align → CR, matched steps. The doc tower is **not** trained (teacher frozen), so this is not the paper's joint baseline. |
| `prune` | Table 6 | Progressive 28→14→7→4→2 vs direct 28→4 (same final shape), full fine-tune with L2 at each stage. |
| `cr` | Table 7 | Before vs after contrastive refinement, for the aligned and the pruned 4-layer students (run `prune` first). |
| `latency` | Table 9, Fig. 3 | Query-encoding latency / throughput and speedup vs the teacher; mean cos(q, d+) per phase. |

"P@100" in the paper is reported here as `recall@100` (~1 positive per query, so precision@100 ≤ 0.01).

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import torch
from model import get_device
from teacher import TEACHER_DEFAULTS as T

DEBUG = True    # debug caches (phase_1_teacher with DEBUG = True). Set False for the full run.
NPROCS = 2      # one process per T4
EXP = "losses"  # "losses" | "oneshot" | "prune" | "cr" | "latency"

# --- inputs: earlier Kaggle notebooks' outputs, attached as inputs (read only). Off Kaggle: the local phase dirs.
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
KAGGLE_IN = "/kaggle/input/notebooks/tensorpanda231"
if ON_KAGGLE:
    DATA_PATH = f"{KAGGLE_IN}/hlm-poc-phase-1-teacher/cache/{MODE}/data.json"
    EMB_DIR = f"{KAGGLE_IN}/hlm-poc-phase-1-teacher/cache/{MODE}/emb/teacher"
    ALIGNED = f"{KAGGLE_IN}/hlm-alignment/runs/{MODE}/align/final"     # phase-2 adapter
else:
    DATA_PATH = f"{ROOT}/cache/{MODE}/data.json"
    EMB_DIR = f"{ROOT}/cache/{MODE}/emb/teacher"
    ALIGNED = f"{ROOT}/runs/{MODE}/align/final"
EXP_DIR = f"{ROOT}/runs/{MODE}/exp"   # outputs: <arm>/final adapters, metrics.json, plots/
# a previous session's saved EXP_DIR (attached as input), copied into EXP_DIR once so its arms are reused
RESUME_FROM = None   # e.g. f"{KAGGLE_IN}/<this notebook>/runs/{MODE}/exp"

# --- losses: phase 2 from scratch per arm (align.ALIGN_DEFAULTS; out_dir is set per arm)
ALIGN = dict(model="Qwen/Qwen3-Embedding-0.6B", dim=T["dim"],
             lora_rank=16, lora_alpha=32, lora_dropout=0.05,
             lora_targets=("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"),
             query_prefix="", max_query_len=32,
             loss="l2", loss_kwargs={},
             lr=3e-4, head_lr=1e-3, weight_decay=0.0, warmup_ratio=0.05,
             epochs=1, batch_size=128,            # per GPU
             grad_clip=1.0, amp=True, seed=0,
             log_every=20, eval_every=200, save_every=500, keep_last=2,
             data_path=DATA_PATH, teacher_emb_dir=EMB_DIR)
LOSS_ARMS = {"align_l2": ("l2", {}),              # arm -> (loss, loss_kwargs); "kl" exists only in run_exp
             "align_kl": ("kl", {"tau": 0.05})}

# --- contrastive refinement, for cr and oneshot (refine.REFINE_DEFAULTS; init_adapter / out_dir set per arm)
REFINE = dict(dim=T["dim"], query_prefix="", max_query_len=32,
              tau=0.05, margin=0.1, n_neg=1, l2_weight=0.1,
              lr=5e-5, head_lr=1e-4, weight_decay=0.0, warmup_ratio=0.05,
              epochs=1, batch_size=64,             # queries per GPU
              grad_clip=1.0, amp=True, seed=0,
              log_every=20, eval_every=200, save_every=500, keep_last=2,
              data_path=DATA_PATH, teacher_emb_dir=EMB_DIR)
# oneshot uses REFINE with l2_weight=0 and enough epochs to match align + CR steps (printed when it runs)

# --- pruning: full fine-tune of every weight at each stage (prune.PRUNE_DEFAULTS; init / n_layers / out_dir per arm)
PRUNE = dict(ffn_keep=0.75,                        # FFN width kept per progressive stage
             dim=T["dim"], query_prefix="", max_query_len=32,
             lr=2e-5, head_lr=1e-4, weight_decay=0.0, warmup_ratio=0.05,
             epochs=1, batch_size=64,              # per GPU
             grad_clip=1.0, amp=True, seed=0,
             log_every=20, eval_every=200, save_every=500, keep_last=2,
             data_path=DATA_PATH, teacher_emb_dir=EMB_DIR)
PRUNE_STAGES = (14, 7, 4, 2)                       # progressive: 28 -> 14 -> 7 -> 4 -> 2
PRUNE_DIRECT = 4                                   # direct: 28 -> 4 in one cut, same final shape as prog_l4

# --- cr: students refined (pruned ones exist after EXP = "prune")
CR_STUDENTS = {"aligned": ALIGNED,
               "prog_l4": f"{EXP_DIR}/prog_l4/final",
               "direct_l4": f"{EXP_DIR}/direct_l4/final"}

# --- latency: name -> (model, max_len, prefix); the first is the speedup reference
LATENCY_MODELS = {"teacher_4b": (T["model"], T["query_max_len"], T["query_prompt"]),
                  "base_0.6b": (ALIGN["model"], 32, ""),          # zero-shot student: cos(q, d+) before alignment
                  "aligned": (ALIGNED, 32, ""),
                  "aligned_cr": (f"{EXP_DIR}/aligned_cr/final", 32, ""),
                  **{f"prog_l{n}": (f"{EXP_DIR}/prog_l{n}/final", 32, "") for n in PRUNE_STAGES},
                  "direct_l4": (f"{EXP_DIR}/direct_l{PRUNE_DIRECT}/final", 32, "")}
LATENCY = dict(n_queries=2000, batch_sizes=(1, 8, 32, 128))

# --- evaluation (evaluation.evaluate_model kwargs)
EVAL = dict(splits=("val", "test"), dim=T["dim"], max_query_len=32, query_prefix="", device=get_device(), dtype="fp16")

if DEBUG:
    for cfg in (ALIGN, REFINE, PRUNE):
        cfg.update(log_every=5, eval_every=20, save_every=20)
    LATENCY.update(n_queries=200)

## 1 · Check inputs
Teacher cache valid (finite, unit norm, in `data.json` order) and the phase-2 adapter matches its dim.

In [ ]:
import os
import shutil
from data import load_json
from evaluation import load_index

if RESUME_FROM and not os.path.exists(EXP_DIR):
    shutil.copytree(RESUME_FROM, EXP_DIR)
    print("resumed from", RESUME_FROM, "->", sorted(os.listdir(EXP_DIR)))
data = load_json(DATA_PATH)
index = load_index(data, EMB_DIR, ("val", "test"))   # (teacher docs, {split: teacher queries})
assert os.path.exists(f"{EMB_DIR}/align.npy"), "align.npy missing: needed by losses / prune / cr"
adapter_cfg = load_json(f"{ALIGNED}/adapter_config.json")
print("phase-2 adapter:", adapter_cfg)
assert adapter_cfg["out_dim"] == T["dim"]
print({k: len(v["ids"]) for k, v in data.items()})

## 2 · Run `EXP`
Trains the missing arms (DDP over `NPROCS` GPUs), evaluates the missing rows, prints the table and shows the plots.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from run_exp import exp_cr, exp_latency, exp_losses, exp_oneshot, exp_prune

if EXP == "losses":
    res = exp_losses(ALIGN, LOSS_ARMS, data, index, EXP_DIR, EVAL, NPROCS)
elif EXP == "oneshot":
    res = exp_oneshot(ALIGN, REFINE, ALIGNED, data, index, EXP_DIR, EVAL, NPROCS)
elif EXP == "prune":
    res = exp_prune(PRUNE, ALIGNED, PRUNE_STAGES, PRUNE_DIRECT, data, index, EXP_DIR, EVAL, NPROCS)
elif EXP == "cr":
    missing = [n for n, d in CR_STUDENTS.items() if not os.path.exists(d)]
    print("skipped (not trained yet, run EXP = 'prune' first):", missing)
    res = exp_cr(REFINE, {n: d for n, d in CR_STUDENTS.items() if n not in missing}, data, index, EXP_DIR, EVAL,
                 NPROCS)
elif EXP == "latency":
    found = {n: m for n, m in LATENCY_MODELS.items() if not m[0].startswith("/") or os.path.exists(m[0])}
    print("skipped (not trained yet):", sorted(set(LATENCY_MODELS) - set(found)))
    res = exp_latency(found, data, index, EXP_DIR, EVAL["device"], T["dim"], **LATENCY)
for fig in res["figs"]:
    display(fig)
plt.close("all")

## 3 · All rows so far
Every experiment's rows in `EXP_DIR/metrics.json`, teacher first; retention = ndcg@10 / teacher ndcg@10.

In [ ]:
from evaluation import summarize

for split in ("val", "test"):
    summarize(EXP_DIR, split)
    print()

## 4 · Saved adapters
Everything under `/kaggle/working` is kept as this notebook's output when you *Save Version*; attach it as an input to a later session to reuse the arms (e.g. `prune` → `cr` → `latency`).

In [ ]:
import glob

for final in sorted(glob.glob(f"{EXP_DIR}/*/final")):
    mb = sum(os.path.getsize(f) for f in glob.glob(f"{final}/*")) / 1e6
    print(f"{final:<70} {mb:8.1f} MB")
print("plots:", sorted(os.listdir(f"{EXP_DIR}/plots")) if os.path.exists(f"{EXP_DIR}/plots") else [])